In [ ]:
from sympy import *

from IPython.display import display, Math, Latex
from sympy.interactive import printing
printing.init_printing(use_latex='mathjax')
platex = lambda A: latex(A,mat_str='pmatrix',mat_delim='')

%matplotlib inline

# Aufgabenstellung

Bestimmen Sie die TSVD bzw. Tikhonov-Regularisierung von

In [ ]:
A = Matrix([[0,8],[-1,0],[0,6]])
Math('A=' + platex(A))

mit

In [ ]:
alfa = Rational(1,4)
Math(r'\alpha =' + latex(alfa))

# Lösung

## TSVD

Wir bestimmen zunächst die Singulärwertzerlegung $A = U\Sigma V^T$, indem wir die Eigenwerte und Eigenvektoren von $A^T A$ und $AA^T$ berechnen.

In [ ]:
ATA = A.T*A

def ew(A):
    l = A.eigenvects()
    l.sort(reverse = True)
    
    sig = []
    ev = []
    for ewev in l:
        n = ewev[1]
        for i in range(n):
            sig.append(sqrt(ewev[0]))
            v = ewev[2][i]
            ev.append(v.T / v.norm(2))
    
    return sig, Matrix(ev).T

sig, V = ew(ATA)

display(Math(r'A^TA = ' + platex(ATA)))
display(Math(r'\sigma_i \in' + latex(sig) + r', \qquad V =' + platex(V)))

In [ ]:
AAT = A*A.T
sig, Ut = ew(AAT)

display(Math(r'AA^T = ' + platex(AAT)))
display(Math(r'\sigma_i \in' + latex(sig) + r', \qquad \tilde{U} =' + platex(Ut)))

Bei der SVD gilt $A = U \Sigma V^T$, also $\Sigma = U^T A V$.
Wir berechnen nun

In [ ]:
S = Ut.T*A*V
display(Math(r'\tilde{\Sigma} = \tilde{U}^T A V = ' + platex(S)))

Negative Einträge auf der Diagonalen können wir beseitigen, indem wir z.B. die entsprechenden Spalten in $\tilde{U}$ mit $-1$ multiplizieren (an der Orthonormalität der Matrix ändert sich nichts), so dass wir schließlich folgende SVD erhalten

In [ ]:
p = min(A.shape)
U = Ut.copy()
for i in range(p):
    if S[i,i] < 0:
        S[i,i] = -S[i,i]
        U[:,i] = -U[:,i]

Math(r'A = U\Sigma V^T = ' + platex(U) + platex(S)  + platex(V.T) + ' = ' + platex(U*S*V.T))

Bei TSVD setzen wir nun alle Singulärwerte $\sigma_k$ auf 0, für die

\begin{align*}
\frac{\sigma_k}{\sigma_1} < \sqrt{\alpha}
\end{align*}
gilt, d.h. aus $\Sigma$ wird $\Sigma_{\alpha}$

In [ ]:
Sa = S.copy()
Sak = S.T.copy()
for i in range(p):
    if Sa[i,i] < sqrt(alfa) * Sa[0,0]:
        Sa[i,i] = 0
        Sak[i,i] = 0
    else:
        Sak[i,i] = 1 / Sa[i,i]

display(Math(r'\Sigma_{\alpha} = ' + platex(Sa)))

und somit erhalten wir die TSVD-Regularisierung

In [ ]:
Aa = U*Sa*V.T
Math(r'A_{\alpha} = U\Sigma_{\alpha} V^T = ' + platex(U) + platex(Sa) + platex(V.T) + ' = ' + platex(Aa))

bzw. als regularisierte Näherung $x_\alpha$ von $x = A^+ b$

In [ ]:
Aak = V*Sak*U.T
Math(r'x_\alpha = A_{\alpha}^+ b = V\Sigma_{\alpha}^+ U^T b = ' + platex(V) + platex(Sak) + platex(U.T) + 'b = ' + platex(Aak) + "b")

Zum Vergleich betrachten wir die ''wahre'' Pseudoinverse

In [ ]:
Math(r'A^+=' + platex(A.pinv()))

## Tikhonov

Die Tikhonov Regularisierung $x_\alpha$ von $x = A^+ b$ zum Parameter $\alpha$ ist gegeben durch

\begin{align*}
x_\alpha = \big(A^T A + \alpha I \big)^{-1}A^T b
\end{align*}
Mit

In [ ]:
Math('A=' + platex(A))

erhalten wir

In [ ]:
ATA = A.T * A
ATAalfa = ATA + alfa * eye(ATA.shape[0])
Math(r'A^TA =' + platex(ATA) + r',\qquad A^T A + \alpha I =' + platex(ATAalfa))

bzw.

In [ ]:
ATAalfainv = ATAalfa.inv()
Math(r'\big(A^T A + \alpha I\big)^{-1} =' + platex(ATAalfainv))

und somit

In [ ]:
Rtik = ATAalfainv * A.T
Math(rf'x_\alpha = \big(A^T A + \alpha I \big)^{{-1}}A^T b = {platex(ATAalfainv)}{platex(A.T)} b = {platex(Rtik)} b \approx {platex(Rtik.evalf(3))} b')

Zum Vergleich betrachten wir noch einmal die ''wahre'' Pseudoinverse

In [ ]:
Math(rf'A^+ = {platex(A.pinv())} \approx {platex(A.pinv().evalf(3))}')